# 8.7 文筆更活潑等於洞察更深嗎？


兩個助手回答2+2：一個只寫4，另一個寫了一大段漂亮文字卻說5。如果評分只偏愛活潑，第二個可能勝出；但對要查算術的使用者，這次更新讓功能變差了。本節的問題是：如何報告文風改善，同時讓讀者看見內容是否受損？

前置是[8.1把風格拆成可觀察項目](https://birdhackor.github.io/tiny-perceptron-vlm/8.1.html)。回顧那裡的做法，答對與比喻貼切各有一個標籤，不必被迫合成一個數字。本節進一步示範：即使你硬算總分，風格與正確性的權重也會改變排序。權重在這裡指評分項目乘上的重要程度，不是模型裡可訓練的數字；相同詞在不同場合的角色要分清楚。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
answers = [
    {"名稱": "準確短答", "風格": 1, "正確": 1},
    {"名稱": "華麗錯答", "風格": 3, "正確": 0},
]
for style_weight in [0.2, 2.0]:
    print("風格權重", style_weight)
    for row in answers:
        total = style_weight * row["風格"] + row["正確"]
        print(row["名稱"], "分項", row["風格"], row["正確"], "總分", round(total, 1))

輸入是人工設計的兩筆分數，風格1到3在這個例子代表越大越活潑，正確1表示答對、0表示答錯。這不是宣稱真實評分者認為每篇華麗文章都得3分。外層迴圈只換總分中風格的係數，內層把同兩筆分數重新計算。`round(total, 1)`把輸出四捨五入到一位小數，方便對照。

係數0.2時，準確短答得1.2，華麗錯答得0.6，前者勝出；係數2.0時，分數變成3.0與6.0，後者勝出。回答內容完全沒有改變，改的只是我們決定如何加總。

圖中每列是一份固定回答，中間保留風格與正確分項；右邊兩欄只是改加總係數，1.2與0.6變成3.0與6.0。排名翻轉時，「華麗錯答」的正確分仍是0，這正是只看總分會漏掉的事。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/multimodal_style_scores.svg")))

這說明總分排名不是自動浮現的「真實品質」，它包含評分者對取捨的選擇。若只報最後的6.0，看結果的人甚至不知道答案是錯的。

正式比較兩個模型時，應保持同一批問題，並排顯示正確率、任務完成率與風格判讀，說明每項有幾筆測試。任務完成率是符合完整要求的比例，例如[8.5](https://birdhackor.github.io/tiny-perceptron-vlm/8.5.html)中算對還要格式正確。一小批題的風格平均提高，可以是值得研究的訊號，但不能單憑它說模型洞察更深；洞察是否有用，要回到它是否提供可查證、能解決問題的新角度。

實跑也提醒我們，連loss都會被答案組成影響。這裡報告的loss是各個有效學習位置的猜錯代價加總後，再除以位置數的平均值；「有效目標」就是[8.3](https://birdhackor.github.io/tiny-perceptron-vlm/8.3.html)中`labels`不等於`-100`、要計入學習的文字片段位置，也包括回答結束標記。8.3的預設短答模型在七題最後檢查的平均代價為8.41357，固定積木句版本則只有0.26805；但是兩者數值都沒有答對。短答七題合計14個有效位置，長版合計308個，所以兩個平均數的分母不同。長版的固定句中，許多位置已容易猜對，這些較小的代價也一起加入平均，可能把平均值壓低，即使真正的算術答案仍錯。不能拿較小的平均數就說算術更好，或把固定句的七次出現升格成洞察更深。要量「有助理解」，還得換題材、讀實際比喻並核對其關係，而不是只數關鍵字。

練習把風格係數設為0.5，先手算兩筆都為1.5，再執行核對。這個平手沒有讓錯答變正確。試著不用總分，用兩句話描述兩個回答各自的長處與失敗，就能看見並排報告為什麼更容易被理解與質疑。
